In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("../../data/wustl_iiot_2021.csv")
df.head()

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,SAppBytes,DAppBytes,TotAppByte,SynAck,RunTime,sTos,SrcJitAct,DstJitAct,Traffic,Target
0,2019-08-19 12:23:28,2019-08-19 12:23:28,192.168.0.20,192.168.0.2,0,59034,502,10,8,18,...,24,20,44,0.001176,0.053037,0,0.000000,0.0,normal,0
1,2019-08-19 15:13:24,2019-08-19 15:13:24,192.168.0.20,192.168.0.2,0,55841,502,10,8,18,...,24,20,44,0.001308,0.052681,0,0.000000,0.0,normal,0
2,2019-08-19 13:41:31,2019-08-19 13:41:31,192.168.0.20,192.168.0.2,0,63774,502,10,8,18,...,24,20,44,0.000690,0.051793,0,0.000000,0.0,normal,0
3,2019-08-19 12:43:19,2019-08-19 12:43:20,209.240.235.92,192.168.0.2,0,61771,80,4,0,4,...,0,0,0,0.000000,0.889555,0,419.338813,0.0,DoS,1
4,2019-08-19 14:49:44,2019-08-19 14:49:48,192.168.0.20,192.168.0.1,3,0,0,14,0,14,...,476,0,476,0.000000,3.500055,0,525.146562,0.0,normal,0


In [51]:
df["StartTime"] = pd.to_datetime(df["StartTime"])
df["LastTime"] = pd.to_datetime(df["LastTime"])

negative_duration_mask = df['StartTime'] > df['LastTime']
df['StartTime'], df['LastTime'] = (np.where(negative_duration_mask, df['LastTime'], df['StartTime']),np.where(negative_duration_mask, df['StartTime'], df['LastTime']),)

df = df.sort_values("StartTime").reset_index(drop=True)
df["nearest_sec"] = df["StartTime"].dt.floor("s")

In [52]:
per_second = (
    df.groupby(["SrcAddr", "DstAddr", "nearest_sec"])
      .agg(
          flow_count=("StartTime", "size"),

          src_pkts=("SrcPkts", "sum"),
          dst_pkts=("DstPkts", "sum"),
          tot_pkts=("TotPkts", "sum"),

          src_bytes=("SrcBytes", "sum"),
          dst_bytes=("DstBytes", "sum"),
          tot_bytes=("TotBytes", "sum"),

          duration_mean=("Dur", "mean"),

          src_rate_mean=("SrcRate", "mean"),
          dst_rate_mean=("DstRate", "mean"),
          rate_mean=("Rate", "mean"),

          src_load_mean=("SrcLoad", "mean"),
          dst_load_mean=("DstLoad", "mean"),

          src_jitter_mean=("SrcJitter", "mean"),
          dst_jitter_mean=("DstJitter", "mean"),

          src_loss=("SrcLoss", "sum"),
          dst_loss=("DstLoss", "sum"),
          loss=("Loss", "sum"),

          tcp_rtt_mean=("TcpRtt", "mean"),
          traffic = ("Traffic", "first"),
          target=("Target", "max")
      )
      .reset_index()
)

In [53]:
feature_cols = ["flow_count","src_pkts","dst_pkts","tot_pkts","src_bytes","dst_bytes","tot_bytes","duration_mean","src_rate_mean",
    "dst_rate_mean","rate_mean","src_load_mean","dst_load_mean","src_jitter_mean","dst_jitter_mean","src_loss","dst_loss",
    "loss","tcp_rtt_mean"]

In [56]:
def resample_pair(group):

    group = group.sort_values("nearest_sec")
    group = group.set_index("nearest_sec")

    full_index = pd.date_range(
        start=group.index.min(),
        end=group.index.max(),
        freq="1s"
    )

    group = group.reindex(full_index)
    group.index.name = "nearest_sec"
    return group

In [ ]:
seq_len = 5

X = []
y = []

for (src, dst), group in per_second.groupby(["SrcAddr", "DstAddr"]):
    group = group.sort_values("nearest_sec")
    resampled_group = resample_pair(group)
    features = resampled_group[feature_cols].values
    labels = resampled_group["target"].values

    if len(resampled_group) < seq_len:
        continue

    for i in range(0,len(resampled_group) - seq_len + 1, seq_len):
        X_seq = features[i:i + seq_len]
        y_seq = labels[i + seq_len - 1]

        X.append(X_seq)
        y.append(y_seq)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X,y, random_state = 42)